# 01: Collaborative Filtering: SVD Matrix Factorization & Top-K Ranking

**Track 11: Recommendation Systems** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Build recommendation engines using Singular Value Decomposition (SVD): User-Item sparse utility matrix, latent factor decomposition, and Top-K recommendation retrieval.


## 1. User-Item Interaction Matrix Ingestion
Loading movie ratings dataset and building sparse user-item interaction matrix.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds

df = load_dataset("movie_ratings")
print(f"Movie Ratings: {len(df)} interactions")
print(df.head())

pivot_df = df.pivot(index="userId", columns="movieId", values="rating").fillna(0)
matrix = pivot_df.values
user_ratings_mean = np.mean(matrix, axis=1)
matrix_demeaned = matrix - user_ratings_mean.reshape(-1, 1)

print(f"Utility Matrix Shape: {matrix.shape} (Users x Movies)")

## 2. Low-Rank Matrix Factorization via SVD
Decompose matrix $R \approx U \Sigma V^T$ with rank $k=10$.

In [ ]:
k_factors = min(10, min(matrix.shape) - 1)
U, sigma, Vt = svds(matrix_demeaned, k=k_factors)
sigma_diag = np.diag(sigma)

predicted_ratings = np.dot(np.dot(U, sigma_diag), Vt) + user_ratings_mean.reshape(-1, 1)
preds_df = pd.DataFrame(predicted_ratings, columns=pivot_df.columns, index=pivot_df.index)

print("Reconstructed Predicted Ratings Sample (Top 5 users, 5 movies):")
print(preds_df.iloc[:5, :5].round(2))